In [0]:
from pyspark.sql import functions as F

# Read cleaned data from the Silver layer
silver_df = spark.read.table("silver_orders")

print("Silver records:", silver_df.count())

display(silver_df)

In [0]:
gold_summary = silver_df.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("total_amount").alias("total_revenue"),
    F.avg("total_amount").alias("average_order_value")
)

display(gold_summary)

In [0]:
revenue_by_product = (
    silver_df
    .groupBy("product")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_product)

In [0]:
daily_sales = (
    silver_df
    .groupBy("order_date")
    .agg(
        F.sum("total_amount").alias("daily_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy("order_date")
)

display(daily_sales)

In [0]:
# Save overall sales metrics
gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_sales_summary")

# Save revenue by product
revenue_by_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_revenue_by_product")

# Save daily sales
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_sales")

print("Gold tables created successfully!")

In [0]:
display(spark.read.table("gold_sales_summary"))
display(spark.read.table("gold_revenue_by_product"))
display(spark.read.table("gold_daily_sales"))